# Agente Analista de Datos: SQL + Gráficas + Respuestas en Lenguaje Natural

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexfazio/InteligenciaArtificial/blob/main/tutoriales/notebooks/text-to-sql/03-agente-analista-sql.ipynb)

Construye un agente analista con tres herramientas: ejecutar SQL, generar gráficas y describir datos. El agente decide qué herramienta usar en cada paso y sintetiza una respuesta final con insights.

In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import os
import sqlite3
import pandas as pd
import json
import io
import base64
import anthropic
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

client = anthropic.Anthropic()

# Base de datos de ventas con 5 tablas
conn = sqlite3.connect(':memory:')
conn.executescript("""
    CREATE TABLE regiones (id INTEGER PRIMARY KEY, nombre TEXT, pais TEXT);
    CREATE TABLE vendedores (
        id INTEGER PRIMARY KEY, nombre TEXT, region_id INTEGER,
        FOREIGN KEY (region_id) REFERENCES regiones(id)
    );
    CREATE TABLE categorias (id INTEGER PRIMARY KEY, nombre TEXT, margen_objetivo REAL);
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY, nombre TEXT, categoria_id INTEGER,
        precio REAL, costo REAL,
        FOREIGN KEY (categoria_id) REFERENCES categorias(id)
    );
    CREATE TABLE ventas (
        id INTEGER PRIMARY KEY, vendedor_id INTEGER, producto_id INTEGER,
        fecha TEXT, cantidad INTEGER, precio_final REAL,
        FOREIGN KEY (vendedor_id) REFERENCES vendedores(id),
        FOREIGN KEY (producto_id) REFERENCES productos(id)
    );
    INSERT INTO regiones VALUES (1,'Norte','España'),(2,'Sur','España'),(3,'Centro','España');
    INSERT INTO categorias VALUES (1,'Electrónica',0.30),(2,'Periféricos',0.45),(3,'Audio',0.40);
    INSERT INTO vendedores VALUES (1,'Ana G.',1),(2,'Luis M.',2),(3,'Sara P.',3),(4,'Carlos R.',1);
    INSERT INTO productos VALUES
        (1,'Laptop Pro',1,1299.99,900.00),(2,'Monitor 4K',1,449.99,310.00),
        (3,'Teclado',2,89.99,45.00),(4,'Mouse',2,39.99,18.00),(5,'Auriculares',3,129.99,75.00);
    INSERT INTO ventas VALUES
        (1,1,1,'2024-01-15',2,2599.98),(2,2,3,'2024-01-20',5,449.95),
        (3,3,5,'2024-02-01',3,389.97),(4,1,2,'2024-02-14',1,449.99),
        (5,4,4,'2024-03-05',10,399.90),(6,2,1,'2024-03-18',1,1299.99),
        (7,3,3,'2024-04-02',8,719.92),(8,1,5,'2024-04-20',2,259.98),
        (9,4,2,'2024-05-10',2,899.98),(10,2,4,'2024-05-25',15,599.85);
""")
conn.commit()
print('BD de ventas con 5 tablas creada.')

In [ ]:
# Esquema para el agente
ESQUEMA = """regiones(id, nombre, pais)
vendedores(id, nombre, region_id -> regiones.id)
categorias(id, nombre, margen_objetivo)
productos(id, nombre, categoria_id -> categorias.id, precio, costo)
ventas(id, vendedor_id -> vendedores.id, producto_id -> productos.id, fecha, cantidad, precio_final)"""

# Definición de herramientas para Claude
HERRAMIENTAS = [
    {
        'name': 'ejecutar_sql',
        'description': 'Ejecuta una consulta SQL SELECT en la base de datos y devuelve los resultados.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'sql': {'type': 'string', 'description': 'Consulta SQL SELECT válida para SQLite'},
                'descripcion': {'type': 'string', 'description': 'Qué datos está obteniendo esta consulta'}
            },
            'required': ['sql', 'descripcion']
        }
    },
    {
        'name': 'generar_grafica',
        'description': 'Genera una gráfica de barras, línea o torta para visualizar datos.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'datos': {'type': 'array', 'items': {'type': 'object'}, 'description': 'Lista de dicts con los datos'},
                'tipo': {'type': 'string', 'enum': ['barras', 'linea', 'torta'], 'description': 'Tipo de gráfica'},
                'columna_x': {'type': 'string', 'description': 'Columna para eje X'},
                'columna_y': {'type': 'string', 'description': 'Columna para eje Y'},
                'titulo': {'type': 'string', 'description': 'Título de la gráfica'}
            },
            'required': ['datos', 'tipo', 'columna_x', 'columna_y', 'titulo']
        }
    },
    {
        'name': 'describir_datos',
        'description': 'Calcula estadísticas descriptivas (media, min, max, etc.) de una columna numérica.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'datos': {'type': 'array', 'items': {'type': 'object'}, 'description': 'Lista de dicts con los datos'},
                'columna': {'type': 'string', 'description': 'Columna numérica a analizar'}
            },
            'required': ['datos', 'columna']
        }
    }
]
print('Herramientas definidas:', [h['name'] for h in HERRAMIENTAS])

In [ ]:
# Implementación de las herramientas
def tool_ejecutar_sql(sql, conn):
    try:
        if not sql.strip().upper().startswith(('SELECT', 'WITH')):
            return 'Error: solo se permiten consultas SELECT'
        df = pd.read_sql_query(sql, conn)
        if df.empty:
            return 'La consulta no devolvió resultados.'
        return df.to_json(orient='records', force_ascii=False)
    except Exception as e:
        return f'Error SQL: {str(e)}'

def tool_generar_grafica(params):
    try:
        df = pd.DataFrame(params['datos'])
        col_x, col_y = params['columna_x'], params['columna_y']
        
        if col_x not in df.columns or col_y not in df.columns:
            return f'Error: columnas disponibles: {list(df.columns)}'
        
        fig, ax = plt.subplots(figsize=(9, 5))
        tipo = params['tipo']
        
        if tipo == 'barras':
            ax.bar(df[col_x].astype(str), df[col_y], color='#4A90D9', edgecolor='white')
            ax.set_xlabel(col_x); ax.set_ylabel(col_y)
            plt.xticks(rotation=30, ha='right')
        elif tipo == 'linea':
            ax.plot(df[col_x].astype(str), df[col_y], marker='o', color='#4A90D9', linewidth=2)
            ax.set_xlabel(col_x); ax.set_ylabel(col_y)
            plt.xticks(rotation=30, ha='right')
        elif tipo == 'torta':
            ax.pie(df[col_y], labels=df[col_x].astype(str), autopct='%1.1f%%')
        
        ax.set_title(params['titulo'], fontsize=13, pad=12)
        plt.tight_layout()
        
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        b64 = base64.b64encode(buf.read()).decode()
        plt.close(fig)
        return f'IMAGEN:{b64}'
    except Exception as e:
        plt.close('all')
        return f'Error gráfica: {str(e)}'

def tool_describir_datos(datos, columna):
    try:
        df = pd.DataFrame(datos)
        if columna not in df.columns:
            return f'Columna no encontrada. Disponibles: {list(df.columns)}'
        serie = pd.to_numeric(df[columna], errors='coerce').dropna()
        stats = {
            'columna': columna, 'count': int(serie.count()),
            'mean': round(float(serie.mean()), 2),
            'median': round(float(serie.median()), 2),
            'std': round(float(serie.std()), 2),
            'min': round(float(serie.min()), 2),
            'max': round(float(serie.max()), 2)
        }
        return json.dumps(stats, ensure_ascii=False)
    except Exception as e:
        return f'Error estadísticas: {str(e)}'

print('Implementaciones de herramientas listas.')

In [ ]:
def agente_analista(pregunta, esquema, conn, max_pasos=5):
    """Bucle agéntico: hasta max_pasos, el agente decide qué herramienta usar."""
    mensajes = [{'role': 'user', 'content': pregunta}]
    pasos = 0
    grafica_b64 = None
    herramientas_usadas = []
    
    system = f"""Eres un analista de datos experto. Base de datos disponible:
{esquema}

Usa las herramientas disponibles para responder la pregunta:
1. ejecutar_sql para obtener datos
2. describir_datos para estadísticas
3. generar_grafica cuando la visualización aporte valor
Luego sintetiza una respuesta clara en español con los hallazgos principales."""
    
    while pasos < max_pasos:
        respuesta = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            system=system,
            tools=HERRAMIENTAS,
            messages=mensajes
        )
        pasos += 1
        mensajes.append({'role': 'assistant', 'content': respuesta.content})
        
        if respuesta.stop_reason == 'end_turn':
            texto = next((b.text for b in respuesta.content if hasattr(b, 'text')), 'Análisis completado.')
            return {'respuesta': texto, 'grafica_b64': grafica_b64, 'pasos': pasos, 'herramientas': herramientas_usadas}
        
        if respuesta.stop_reason == 'tool_use':
            resultados = []
            for bloque in respuesta.content:
                if bloque.type != 'tool_use':
                    continue
                herramientas_usadas.append(bloque.name)
                print(f'  → Usando: {bloque.name}')
                
                if bloque.name == 'ejecutar_sql':
                    resultado = tool_ejecutar_sql(bloque.input['sql'], conn)
                elif bloque.name == 'generar_grafica':
                    resultado = tool_generar_grafica(bloque.input)
                    if resultado.startswith('IMAGEN:'):
                        grafica_b64 = resultado[7:]
                        resultado = 'Gráfica generada exitosamente.'
                elif bloque.name == 'describir_datos':
                    resultado = tool_describir_datos(bloque.input['datos'], bloque.input['columna'])
                else:
                    resultado = f'Herramienta {bloque.name} no encontrada'
                
                resultados.append({'type': 'tool_result', 'tool_use_id': bloque.id, 'content': resultado})
            
            mensajes.append({'role': 'user', 'content': resultados})
    
    return {'respuesta': 'Límite de pasos alcanzado.', 'grafica_b64': grafica_b64, 'pasos': pasos, 'herramientas': herramientas_usadas}

print('Agente analista definido.')

In [ ]:
# Demo 1: Vendedor con más ventas
from IPython.display import Image, display

pregunta = '¿Cuál es el vendedor con más ventas totales? Muéstrame una comparación.'
print(f'Pregunta: {pregunta}\n')

resultado = agente_analista(pregunta, ESQUEMA, conn)

print(f'\nHerramientas usadas: {resultado["herramientas"]}')
print(f'Pasos ejecutados: {resultado["pasos"]}')
print(f'\nRespuesta:\n{resultado["respuesta"]}')

if resultado['grafica_b64']:
    img_bytes = base64.b64decode(resultado['grafica_b64'])
    display(Image(data=img_bytes))

In [ ]:
# Demo 2: Tendencia mensual de ventas
pregunta2 = '¿Cómo ha evolucionado el total de ventas mes a mes?'
print(f'Pregunta: {pregunta2}\n')

resultado2 = agente_analista(pregunta2, ESQUEMA, conn)

print(f'Herramientas: {resultado2["herramientas"]}')
print(f'\nRespuesta:\n{resultado2["respuesta"]}')

if resultado2['grafica_b64']:
    img_bytes = base64.b64decode(resultado2['grafica_b64'])
    display(Image(data=img_bytes))

In [ ]:
# Demo 3: Análisis de márgenes por categoría
pregunta3 = '¿Qué categoría de productos genera mayor margen de ganancia real?'
print(f'Pregunta: {pregunta3}\n')

resultado3 = agente_analista(pregunta3, ESQUEMA, conn)

print(f'Herramientas: {resultado3["herramientas"]}')
print(f'\nRespuesta:\n{resultado3["respuesta"]}')

if resultado3['grafica_b64']:
    img_bytes = base64.b64decode(resultado3['grafica_b64'])
    display(Image(data=img_bytes))

## Resumen

El agente analista combina tres herramientas en un bucle agéntico:

| Herramienta | Cuándo la usa |  
|-------------|---------------|
| `ejecutar_sql` | Siempre — para obtener datos |
| `describir_datos` | Cuando necesita estadísticas numéricas |
| `generar_grafica` | Cuando la visualización ayuda a entender tendencias o comparaciones |

El límite de 5 pasos garantiza que el agente siempre termina y devuelve una respuesta.

**Siguiente**: `04-text-to-sql-produccion.ipynb` — Seguridad, caché y casos empresariales